In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

base_path = "/content/drive/MyDrive/files"

folders = [f for f in os.listdir(base_path) if f.startswith("S")]

filenumber = ['03','04','07','08','11','12']
data = []

for folder in sorted(folders):
    folder_path = os.path.join(base_path, folder)

    if not os.path.isdir(folder_path):
        continue

    files = os.listdir(folder_path)

    for f in files:
        if f.lower().endswith(".edf"):

            run_part = f.split("R")[-1].split(".")[0]

            if run_part in filenumber:
                full_path = os.path.join(folder_path, f)
                data.append(full_path)

# Print final single list
print(data)
print("\nTotal files:", len(data))


['/content/drive/MyDrive/files/S071/S071R12.edf', '/content/drive/MyDrive/files/S071/S071R04.edf', '/content/drive/MyDrive/files/S071/S071R03.edf', '/content/drive/MyDrive/files/S071/S071R11.edf', '/content/drive/MyDrive/files/S071/S071R08.edf', '/content/drive/MyDrive/files/S071/S071R07.edf', '/content/drive/MyDrive/files/S072/S072R12.edf', '/content/drive/MyDrive/files/S072/S072R04.edf', '/content/drive/MyDrive/files/S072/S072R08.edf', '/content/drive/MyDrive/files/S072/S072R11.edf', '/content/drive/MyDrive/files/S072/S072R07.edf', '/content/drive/MyDrive/files/S072/S072R03.edf', '/content/drive/MyDrive/files/S074/S074R07.edf', '/content/drive/MyDrive/files/S074/S074R04.edf', '/content/drive/MyDrive/files/S074/S074R12.edf', '/content/drive/MyDrive/files/S074/S074R03.edf', '/content/drive/MyDrive/files/S074/S074R11.edf', '/content/drive/MyDrive/files/S074/S074R08.edf', '/content/drive/MyDrive/files/S076/S076R12.edf', '/content/drive/MyDrive/files/S076/S076R04.edf', '/content/drive/MyD

In [4]:
!pip install mne

In [5]:
import tensorflow as tf
if tf.test.gpu_device_name():
    print('Default GPU Device: {}'.format(tf.test.gpu_device_name()))
    !nvidia-smi
else:
    print("Please install GPU version of TF")

Please install GPU version of TF


In [6]:


import mne

raw_list = []

for path in data:
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    raw_list.append(raw)

print("Total raw files loaded:", len(raw_list))

/tmp/ipykernel_5914/2038446953.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
/tmp/ipykernel_5914/2038446953.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
/tmp/ipykernel_5914/2038446953.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
/tmp/ipykernel_5914/2038446953.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
/tmp/ipykernel_5914/2038446953.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
/tmp/ipykernel_5914/2038446953.py:6: RuntimeWarning: Limited 1 annotation(s) tha

Total raw files loaded: 222


In [7]:
import numpy as np
t0_all = []
t1_all = []
t2_all = []

for raw in raw_list:

    # 1️⃣ Resample
    raw = raw.copy().resample(160, npad="auto")

    # 2️⃣ Bandpass
    raw = raw.filter(
        l_freq=8.,
        h_freq=30.,
        fir_design='firwin',
        verbose=False
    )

    # 3️⃣ Events
    events, event_dict = mne.events_from_annotations(raw, verbose=False)

    # 4️⃣ Epochs
    epochs = mne.Epochs(
        raw,
        events,
        event_id={'T0': 1, 'T1': 2, 'T2': 3},
        tmin=0.5,
        tmax=2.5,
        baseline=None,
        preload=True,
        verbose=False
    )

    # 5️⃣ Extract numpy data
    data = epochs.get_data()   # shape: (trials, channels, samples)

    # 6️⃣ Normalize per trial (VERY IMPORTANT)
    data = (data - data.mean(axis=2, keepdims=True)) / \
           (data.std(axis=2, keepdims=True) + 1e-8)

    # 7️⃣ Split back per class
    try:
        t0_all.append(data[epochs.events[:,2] == 1])
    except:
        pass

    try:
        t1_all.append(data[epochs.events[:,2] == 2])
    except:
        pass

    try:
        t2_all.append(data[epochs.events[:,2] == 3])
    except:
        pass


# 8️⃣ Concatenate all subjects
t0 = np.concatenate(t0_all, axis=0)
t1 = np.concatenate(t1_all, axis=0)
t2 = np.concatenate(t2_all, axis=0)

print("T0:", t0.shape)
print("T1:", t1.shape)
print("T2:", t2.shape)

Sampling frequency of the instance is already 160.0, returning unmodified.
Sampling frequency of the instance is already 160.0, returning unmodified.
Sampling frequency of the instance is already 160.0, returning unmodified.
Sampling frequency of the instance is already 160.0, returning unmodified.
Sampling frequency of the instance is already 160.0, returning unmodified.
Sampling frequency of the instance is already 160.0, returning unmodified.
Sampling frequency of the instance is already 160.0, returning unmodified.
Sampling frequency of the instance is already 160.0, returning unmodified.
Sampling frequency of the instance is already 160.0, returning unmodified.
Sampling frequency of the instance is already 160.0, returning unmodified.
Sampling frequency of the instance is already 160.0, returning unmodified.
Sampling frequency of the instance is already 160.0, returning unmodified.
Sampling frequency of the instance is already 160.0, returning unmodified.
Sampling frequency of the

In [8]:
!pip install pyriemann

In [9]:
# Your channel list
ch_names_raw = ['Fc5.', 'Fc3.', 'Fc1.', 'Fcz.', 'Fc2.', 'Fc4.', 'Fc6.',
                'C5..', 'C3..', 'C1..', 'Cz..', 'C2..', 'C4..', 'C6..',
                'Cp5.', 'Cp3.', 'Cp1.', 'Cpz.', 'Cp2.', 'Cp4.', 'Cp6.',
                'Fp1.', 'Fpz.', 'Fp2.', 'Af7.', 'Af3.', 'Afz.', 'Af4.', 'Af8.',
                'F7..', 'F5..', 'F3..', 'F1..', 'Fz..', 'F2..', 'F4..', 'F6..', 'F8..',
                'Ft7.', 'Ft8.', 'T7..', 'T8..', 'T9..', 'T10.', 'Tp7.', 'Tp8.',
                'P7..', 'P5..', 'P3..', 'P1..', 'Pz..', 'P2..', 'P4..', 'P6..', 'P8..',
                'Po7.', 'Po3.', 'Poz.', 'Po4.', 'Po8.',
                'O1..', 'Oz..', 'O2..', 'Iz..']

# Clean: remove dots and make uppercase
ch_names = [ch.replace('.', '').upper() for ch in ch_names_raw]

print(ch_names)

['FC5', 'FC3', 'FC1', 'FCZ', 'FC2', 'FC4', 'FC6', 'C5', 'C3', 'C1', 'CZ', 'C2', 'C4', 'C6', 'CP5', 'CP3', 'CP1', 'CPZ', 'CP2', 'CP4', 'CP6', 'FP1', 'FPZ', 'FP2', 'AF7', 'AF3', 'AFZ', 'AF4', 'AF8', 'F7', 'F5', 'F3', 'F1', 'FZ', 'F2', 'F4', 'F6', 'F8', 'FT7', 'FT8', 'T7', 'T8', 'T9', 'T10', 'TP7', 'TP8', 'P7', 'P5', 'P3', 'P1', 'PZ', 'P2', 'P4', 'P6', 'P8', 'PO7', 'PO3', 'POZ', 'PO4', 'PO8', 'O1', 'OZ', 'O2', 'IZ']


In [10]:
motor_channels = [
    'FC1', 'FC3', 'FC5', 'FCZ', 'FC2', 'FC4', 'FC6',
    'C1',  'C3',  'C5',  'CZ',  'C2',  'C4',  'C6',
    'CP1', 'CP3', 'CP5', 'CPZ', 'CP2', 'CP4', 'CP6'
]

In [11]:
motor_idx = [ch_names.index(ch) for ch in motor_channels]

print("Motor Channel Indices:", motor_idx)
t0.shape

Motor Channel Indices: [2, 1, 0, 3, 4, 5, 6, 9, 8, 7, 10, 11, 12, 13, 16, 15, 14, 17, 18, 19, 20]


(3365, 64, 321)

In [12]:
t0_motor = t0[:, motor_idx, :]
t1_motor = t1[:, motor_idx, :]
t2_motor = t2[:, motor_idx, :]

print("New shape:", t0_motor.shape)

New shape: (3365, 21, 321)


In [13]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import copy

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from torch.utils.data import TensorDataset, DataLoader

In [14]:
y0 = np.zeros(len(t0))
y1 = np.ones(len(t1))
y2 = np.ones(len(t2)) * 2

In [15]:
X = np.concatenate([t0, t1, t2], axis=0)
y = np.concatenate([y0, y1, y2], axis=0)
X.shape
X.shape

(6730, 64, 321)

In [16]:
# spliting data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (5384, 64, 321)
Test shape: (1346, 64, 321)


In [17]:
#normalising
mean = X_train.mean(axis=(0,2), keepdims=True)
std = X_train.std(axis=(0,2), keepdims=True)

X_train = (X_train - mean) / (std + 1e-6)
X_test  = (X_test - mean) / (std + 1e-6)

In [18]:
train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)

test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [19]:
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.classification import MDM
from mne.decoding import CSP
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

X_norm = (X - X.mean(axis=2, keepdims=True)) / (X.std(axis=2, keepdims=True) + 1e-8)
# CSP wrapper that handles 3-class via one-vs-rest
def csp_pipe(clf):
    return make_pipeline(
        CSP(n_components=8, log=True, reg=1e-2),
        OneVsRestClassifier(clf)
    )

models = {
    "CSP + LDA"    : csp_pipe(LinearDiscriminantAnalysis()),
    "CSP + SVM"    : csp_pipe(SVC(kernel='rbf', C=10, gamma='scale')),
    "CSP + LR"     : csp_pipe(LogisticRegression(max_iter=2000)),
    "CSP + RF"     : csp_pipe(RandomForestClassifier(n_estimators=200, random_state=42)),
    "Riemann + LDA": make_pipeline(Covariances('oas'), TangentSpace(), LinearDiscriminantAnalysis()),
    "Riemann + SVM": make_pipeline(Covariances('oas'), TangentSpace(), SVC(kernel='rbf', C=10, gamma='scale')),
    "Riemann + LR" : make_pipeline(Covariances('oas'), TangentSpace(), LogisticRegression(max_iter=2000)),
    "Riemann + RF" : make_pipeline(Covariances('oas'), TangentSpace(), RandomForestClassifier(n_estimators=200, random_state=42)),
    "Riemann MDM"  : make_pipeline(Covariances('oas'), MDM()),
}

for name, model in models.items():
    try:
        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))
        print(f"{name:<20} {acc:.4f}")
    except Exception as e:
        print(f"{name:<20} ERROR: {e}")

Computing rank from data with rank=None
    Using tolerance 1.1e+02 (2.2e-16 eps * 64 dim * 7.9e+15  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=0.0 covariance using SHRINKAGE
Done.
Estimating class=1.0 covariance using SHRINKAGE
Done.
Estimating class=2.0 covariance using SHRINKAGE
Done.
CSP + LDA            0.5104
Computing rank from data with rank=None
    Using tolerance 1.1e+02 (2.2e-16 eps * 64 dim * 7.9e+15  max singular value)
    Estimated rank (data): 64
    data: rank 64 computed from 64 data channels with 0 projectors
Reducing data rank from 64 -> 64
Estimating class=0.0 covariance using SHRINKAGE
Done.
Estimating class=1.0 covariance using SHRINKAGE
Done.
Estimating class=2.0 covariance using SHRINKAGE
Done.
CSP + SVM            0.5453
Computing rank from data with rank=None
    Using tolerance 1.1e+02 (2.2e-16 eps * 64 dim * 7.9e+15  max singular valu